<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/transpose_shared.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


from numba.core.types import float32
from numba import cuda
import cupy as cp
import numpy as np

TILE_DIM = 32

@cuda.jit
def transpose (input_mat, output_mat):
  n,m = output_mat.shape

  col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y

  if row < n and col < m:
    output_mat[row,col] = input_mat[col,row]



def main():

  m = 10000
  n = 2000

  device = cp.cuda.Device()

  input_mat = cp.random.uniform(-1, 1, size=(m, n), dtype=cp.float32)
  output_mat = cp.zeros((n, m), dtype= cp.float32)

  #launch configuration
  # no. of elements in smem = no. of threads in block
  threads_per_block = (TILE_DIM, TILE_DIM)
  blocks_per_grid = (
      (m + (TILE_DIM - 1)) // TILE_DIM,
      (n + (TILE_DIM - 1)) // TILE_DIM
  )

  transpose[blocks_per_grid,threads_per_block] (input_mat,output_mat)

  cuda.synchronize()

  input_mat_np = cp.asnumpy(input_mat)
  output_mat_np = cp.asnumpy(output_mat)

  np.testing.assert_allclose(output_mat_np, input_mat_np.T)

  print ("input matrix")
  print (input_mat_np)
  print ("output matrix")
  print (output_mat_np)

main()

input matrix
[[ 0.6274344  -0.7078728   0.85301864 ... -0.92146486  0.31995964
   0.10753953]
 [ 0.5722946   0.8125     -0.1869874  ... -0.10614002  0.4440416
  -0.6349838 ]
 [ 0.4853542   0.7125124   0.4107889  ... -0.8529893  -0.19529492
  -0.3460511 ]
 ...
 [-0.8416783  -0.33530563  0.754472   ... -0.79783314 -0.6785128
   0.73861957]
 [ 0.21972644  0.24928021 -0.6664187  ...  0.554613    0.21879816
   0.80637205]
 [ 0.17033327 -0.93492293  0.668031   ... -0.57505834  0.6778636
   0.7335881 ]]
output matrix
[[ 0.6274344   0.5722946   0.4853542  ... -0.8416783   0.21972644
   0.17033327]
 [-0.7078728   0.8125      0.7125124  ... -0.33530563  0.24928021
  -0.93492293]
 [ 0.85301864 -0.1869874   0.4107889  ...  0.754472   -0.6664187
   0.668031  ]
 ...
 [-0.92146486 -0.10614002 -0.8529893  ... -0.79783314  0.554613
  -0.57505834]
 [ 0.31995964  0.4440416  -0.19529492 ... -0.6785128   0.21879816
   0.6778636 ]
 [ 0.10753953 -0.6349838  -0.3460511  ...  0.73861957  0.80637205
   0.733588

In [ ]:
%%writefile transpose-naive.py



from numba.core.types import float32
from numba import cuda
import cupy as cp
import numpy as np

TILE_DIM = 32

@cuda.jit
def transpose (input_mat, output_mat):
  n,m = output_mat.shape

  col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y

  if row < n and col < m:
    output_mat[row,col] = input_mat[col,row]



def main():

  m = 10000
  n = 2000

  device = cp.cuda.Device()

  input_mat = cp.random.uniform(-1, 1, size=(m, n), dtype=cp.float32)
  output_mat = cp.zeros((n, m), dtype= cp.float32)

  #launch configuration
  # no. of elements in smem = no. of threads in block
  threads_per_block = (TILE_DIM, TILE_DIM)
  blocks_per_grid = (
      (m + (TILE_DIM - 1)) // TILE_DIM,
      (n + (TILE_DIM - 1)) // TILE_DIM
  )

  transpose[blocks_per_grid,threads_per_block] (input_mat,output_mat)

  cuda.synchronize()

  input_mat_np = cp.asnumpy(input_mat)
  output_mat_np = cp.asnumpy(output_mat)

  np.testing.assert_allclose(output_mat_np, input_mat_np.T)

  print ("input matrix")
  print (input_mat_np)
  print ("output matrix")
  print (output_mat_np)

main()

Overwriting transpose-naive.py


In [ ]:
!ncu --set full -f -o transpose-naive-rep python3 transpose-naive.py

==PROF== Connected to process 13284 (/usr/bin/python3.13)
==PROF== Profiling "generate_seed_pseudo" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "gen_sequenced" - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_random_x_mod_1" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_scale" - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "transpose[abi:v1,cw51cXTLSUwv..." - 4: 0%....50%....100% - 33 passes
input matrix
[[-0.69263446 -0.45430768 -0.06054294 ...  0.0891453   0.636927
   0.6701288 ]
 [ 0.21804452 -0.5646648  -0.7947124  ... -0.81926644  0.49745297
  -0.10657394]
 [ 0.31051004 -0.38033938 -0.16354704 ... -0.80350405  0.5285988
  -0.17110682]
 ...
 [-0.9273031  -0.9121156  -0.94763434 ...  0.66092074 -0.8730683
  -0.5760312 ]
 [ 0.79187405  0.86293995  0.17215872 ... -0.26965767 -0.9236954
  -0.01014924]
 [ 0.86743593 -0.7839098   0.36725104 ...  0.3474853   0.60301065
  -0.30587035]]
output matrix
[[-0.69263446  0.21804452  0.31051004 ... -0.9273

In [ ]:
print ("Shared..")

AssertionError: 
Not equal to tolerance rtol=1e-05, atol=1e-05

Mismatched elements: 10540709 / 12000000 (87.8%)
First 5 mismatches are at indices:
 [0, 1216]: -0.0009309649467468262 (ACTUAL), 0.0 (DESIRED)
 [0, 1217]: -0.41995227336883545 (ACTUAL), 0.0 (DESIRED)
 [0, 1218]: 0.3293708562850952 (ACTUAL), 0.0 (DESIRED)
 [0, 1219]: 0.7324951887130737 (ACTUAL), 0.0 (DESIRED)
 [0, 1220]: -0.42522192001342773 (ACTUAL), 0.0 (DESIRED)
Max absolute difference among violations: 0.9999999
Max relative difference among violations: inf
 ACTUAL: array([[-0.178849, -0.253963, -0.746968, ...,  0.149027,  0.748516,
        -0.891543],
       [-0.947755, -0.177751, -0.589097, ...,  0.262828,  0.719924,...
 DESIRED: array([[-0.178849, -0.253963, -0.746968, ...,  0.      ,  0.      ,
         0.      ],
       [-0.947755, -0.177751, -0.589097, ...,  0.      ,  0.      ,...

In [ ]:
%%writefile transpose-shared.py

from numba import cuda
import cupy as cp
import numpy as np

TILE_DIM = 32
TILE_DIM_ONE = TILE_DIM + 1

@cuda.jit
def transpose(input_mat, output_mat):
    n, m = output_mat.shape
    tile = cuda.shared.array((TILE_DIM, TILE_DIM_ONE), dtype=cp.float32)

    # Phase 1
    # copy from input_mat (gmem) to tile (smem)
    in_col = cuda.threadIdx.x + cuda.blockIdx.y * cuda.blockDim.x # type: ignore
    in_row = cuda.threadIdx.y + cuda.blockIdx.x * cuda.blockDim.y # type: ignore
    if in_row < m and in_col < n:
        tile[cuda.threadIdx.y, cuda.threadIdx.x] = input_mat[in_row, in_col]

    cuda.syncthreads()

    # Phase 2
    # copy from tile (smem) to output_mat (gmem)
    out_col = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x # type: ignore
    out_row = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y # type: ignore
    if out_row < n and out_col < m:
        output_mat[out_row, out_col] = tile[cuda.threadIdx.x, cuda.threadIdx.y]


def main():
    m = 10000
    n = 1200

    input_mat = cp.random.uniform(-1, 1, (m, n), dtype=cp.float32)
    output_mat = cp.zeros((n, m), dtype=cp.float32)

    # launch configuration
    threads_per_block = (TILE_DIM, TILE_DIM)
    blocks_per_grid = (
        (m + threads_per_block[0] - 1) // threads_per_block[0],
        (n + threads_per_block[1] - 1) // threads_per_block[1],
    )
    transpose[blocks_per_grid, threads_per_block](input_mat, output_mat)
    cuda.synchronize()

    input_mat_np = cp.asnumpy(input_mat)
    output_mat_np = cp.asnumpy(output_mat)
    np.testing.assert_allclose(input_mat_np.T, output_mat_np, rtol=1e-5, atol=1e-5)

main()

In [ ]:
!ncu --set full -f -o transpose-shared-rep python3 transpose-shared.py